# Fine-tuning QLoRA d'un petit modèle sur le kabyle

**Kabyle LLM Benchmark, phase 2, approche 3.** Ce notebook entraîne un adaptateur LoRA sur un modèle quantifié en 4 bits (QLoRA), à partir des paires français–kabyle **décontaminées** produites par `pipelines/prepare.py`. Il exporte ensuite le modèle au format GGUF pour le servir localement avec Ollama et l'évaluer dans le labo.

## Avant de lancer : coûts et durées estimés

Ce sont des estimations grossières, non mesurées. Hypothèses : modèle de 4 milliards de paramètres en 4 bits, LoRA r=16, environ 60 tokens par exemple.

| Plateforme | GPU | Coût | 60 000 exemples, 1 époque | 185 000 exemples (tout), 1 époque |
|---|---|---|---|---|
| Google Colab (gratuit) | T4 16 Go (non garanti) | 0 € | ≈ 1 à 1,5 h | ≈ 3 à 4 h, au risque d'une coupure de session |
| Kaggle (gratuit, ≈ 30 h GPU/semaine) | T4 ×2 ou P100 | 0 € | ≈ 1 à 1,5 h | ≈ 3 à 4 h |
| RunPod / Vast.ai | RTX 4090 | ≈ 0,34 à 0,90 $/h | ≈ 10 à 20 min | ≈ 30 à 60 min, soit moins de 1 $ |

Recommandation : commencer sur **Kaggle ou Colab gratuit avec 60 000 exemples**. Ne louer un GPU que si le gain mesuré justifie d'entraîner sur tout le corpus.

## Règle anti-contamination

N'utiliser **que** `sft_train.jsonl` et `sft_valid.jsonl`, générés par `pipelines/prepare.py` **après** la saisie du jeu de test. Ce script retire toute phrase qui recoupe un test, et `pipelines/check_contamination.py` le vérifie. Ne jamais charger un fichier de `data/test/` dans ce notebook.

## Étapes

1. Dans le labo (sur le PC) : `docker compose run --rm api python -m pipelines.prepare`, puis `… pipelines.check_contamination`.
2. Activer le GPU : Colab via *Exécution → Modifier le type d'exécution → T4* ; Kaggle via *Settings → Accelerator → GPU*.
3. Exécuter les cellules dans l'ordre, puis téléverser `sft_train.jsonl` et `sft_valid.jsonl` quand c'est demandé.
4. Télécharger le fichier `.gguf` produit et suivre `notebooks/README.md` pour le servir avec Ollama.

In [ ]:
%%capture
# Unsloth : QLoRA environ 2 fois plus rapide et moins gourmand en mémoire
!pip install -q unsloth

In [ ]:
# ── Configuration ──────────────────────────────────────────
BASE_MODEL = "unsloth/Qwen3.5-4B"  # même modèle que « qwen3.5-4b » dans le labo
OUTPUT_NAME = "kab-qwen3.5-4b"
MAX_SEQ_LENGTH = 512        # les paires Tatoeba sont courtes
MAX_TRAIN_EXAMPLES = 60_000 # None = tout le corpus (≈ 185 000 exemples)
EPOCHS = 1
LEARNING_RATE = 2e-4
LORA_RANK = 16
SEED = 13

In [ ]:
# ── Données : téléversement (Colab) ou jeu de données Kaggle ──
import os

DATA_DIR = "."
if os.path.exists("/kaggle/input"):
    # Sur Kaggle : ajouter un « Dataset » contenant les deux fichiers, puis ajuster ce chemin
    candidates = [r for r, _, files in os.walk("/kaggle/input") if "sft_train.jsonl" in files]
    DATA_DIR = candidates[0] if candidates else "."
else:
    try:
        from google.colab import files
        if not os.path.exists("sft_train.jsonl"):
            files.upload()  # choisir sft_train.jsonl et sft_valid.jsonl
    except ImportError:
        pass

TRAIN_FILE = os.path.join(DATA_DIR, "sft_train.jsonl")
VALID_FILE = os.path.join(DATA_DIR, "sft_valid.jsonl")
assert os.path.exists(TRAIN_FILE) and os.path.exists(VALID_FILE), "fichiers d'entraînement introuvables"
print(TRAIN_FILE, VALID_FILE)

In [ ]:
# ── Modèle de base en 4 bits + adaptateur LoRA ─────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
# ── Mise en forme avec le gabarit de conversation du modèle ─
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": TRAIN_FILE, "valid": VALID_FILE})
if MAX_TRAIN_EXAMPLES:
    ds["train"] = ds["train"].shuffle(seed=SEED).select(range(min(MAX_TRAIN_EXAMPLES, len(ds["train"]))))

def to_text(batch):
    # enable_thinking=False : réponses directes, comme dans le labo (think: false)
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, enable_thinking=False)
                     for m in batch["messages"]]}

ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"])

In [ ]:
# ── Entraînement ───────────────────────────────────────────
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["valid"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_ratio=0.03,
        optim="adamw_8bit",
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=500,
        save_steps=500,
        save_total_limit=2,
        output_dir="checkpoints",
        seed=SEED,
        report_to="none",
    ),
)
# La perte ne porte que sur les réponses (pas sur les consignes)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# En cas de coupure de session : resume_from_checkpoint=True reprend au dernier point de sauvegarde
stats = trainer.train()
print(stats)

In [ ]:
# ── Vérification rapide sur le jeu de VALIDATION (jamais le jeu de test) ──
FastLanguageModel.for_inference(model)
import json

with open(VALID_FILE, encoding="utf-8") as f:
    samples = [json.loads(line) for _, line in zip(range(5), f)]
for s in samples:
    prompt = tokenizer.apply_chat_template(s["messages"][:1], tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("CONSIGNE :", s["messages"][0]["content"])
    print("MODÈLE   :", answer.strip())
    print("ATTENDU  :", s["messages"][1]["content"], "\n")

In [ ]:
# ── Export : adaptateur LoRA + modèle GGUF (Q4_K_M) pour Ollama ──
model.save_pretrained(f"{OUTPUT_NAME}-lora")
tokenizer.save_pretrained(f"{OUTPUT_NAME}-lora")
model.save_pretrained_gguf(OUTPUT_NAME, tokenizer, quantization_method="q4_k_m")

import glob
print(glob.glob(f"{OUTPUT_NAME}*/*.gguf") + glob.glob(f"{OUTPUT_NAME}*.gguf"))
print("Télécharger le .gguf (et le Modelfile s'il a été généré), puis suivre notebooks/README.md")